# Network Traffic Classification Using Transformer-Enhanced LSTM Models

This notebook is the complete experiment workspace for the project. It can run CIC-Darknet2020, UNSW-NB15, or both datasets. It trains the Simple LSTM baseline and the Transformer-Enhanced LSTM model using the same preprocessing, split, class weighting, callbacks, and evaluation metrics.

## 1. Setup

This cell locates the project root and imports the reusable source-code pipeline. Keep the notebook in `notebooks/` and the source files in `src/`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from preprocessing import prepare_dataset
from train import run_project

PROJECT_ROOT

## 2. Choose Experiment Settings

Set `RUN_MODE` to one of: `cic`, `unsw`, or `all`. The default epoch count is 20. You may set `EPOCHS = 50`; EarlyStopping monitors validation loss and restores the best weights, so training stops when validation performance stops improving.

In [ ]:
RUN_MODE = 'cic'  # options: 'cic', 'unsw', 'all'
EPOCHS = 20
BATCH_SIZE = 256
MAX_SAMPLES = None  # use a small number only for quick testing
CLASS_WEIGHT_MODE = 'sqrt'  # options: 'sqrt', 'balanced', 'none'
SAVE_MODELS = False

## 3. Preview Dataset Preprocessing

This preview checks that the selected dataset(s) load correctly, labels are encoded, features are scaled/encoded, and tensors are reshaped for LSTM input. It uses a small sample for speed and does not affect the final training run.

In [ ]:
preview_datasets = ['cic', 'unsw'] if RUN_MODE == 'all' else [RUN_MODE]

for dataset_key in preview_datasets:
    data = prepare_dataset(dataset=dataset_key, data_dir=PROJECT_ROOT / 'dataset', max_samples=1000)
    print(f'\nDataset: {data.dataset_name}')
    print('Label column:', data.label_column)
    print('Input shape:', data.input_shape)
    print('Classes:', data.class_names)
    print('Train tensor:', data.X_train.shape)
    print('Validation tensor:', data.X_val.shape)
    print('Test tensor:', data.X_test.shape)
    print('Class distribution:', data.class_distribution)

## 4. Train and Evaluate Models

This cell trains both models on the selected dataset(s). It does not delete old results. Single-dataset runs save dataset-specific files, and `RUN_MODE = 'all'` also writes the combined required files.

In [ ]:
if RUN_MODE not in {'cic', 'unsw', 'all'}:
    raise ValueError("RUN_MODE must be one of: 'cic', 'unsw', 'all'")

experiments = run_project(
    dataset='cic' if RUN_MODE == 'all' else RUN_MODE,
    all_datasets=(RUN_MODE == 'all'),
    data_dir=PROJECT_ROOT / 'dataset',
    results_dir=PROJECT_ROOT / 'results',
    figures_dir=PROJECT_ROOT / 'figures',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    max_samples=MAX_SAMPLES,
    class_weight_mode=CLASS_WEIGHT_MODE,
    save_models=SAVE_MODELS,
)

list(experiments.keys())

## 5. Display Metrics

For `RUN_MODE = 'cic'`, read `results/cic_darknet2020_metrics.txt`. For `RUN_MODE = 'unsw'`, read `results/unsw_nb15_metrics.txt`. For `RUN_MODE = 'all'`, read the combined `results/metrics.txt`.

In [ ]:
metrics_file = {
    'cic': PROJECT_ROOT / 'results' / 'cic_darknet2020_metrics.txt',
    'unsw': PROJECT_ROOT / 'results' / 'unsw_nb15_metrics.txt',
    'all': PROJECT_ROOT / 'results' / 'metrics.txt',
}[RUN_MODE]

print(metrics_file)
print(metrics_file.read_text(encoding='utf-8'))

## 6. Display Figures

The notebook displays the relevant confusion matrix, accuracy curve, loss curve, and comparison graph for the selected run.

In [ ]:
from IPython.display import Image, display

prefix = {
    'cic': 'cic_darknet2020_',
    'unsw': 'unsw_nb15_',
    'all': '',
}[RUN_MODE]

figure_paths = [
    PROJECT_ROOT / 'results' / f'{prefix}confusion_matrix.png',
    PROJECT_ROOT / 'figures' / f'{prefix}accuracy_curve.png',
    PROJECT_ROOT / 'figures' / f'{prefix}loss_curve.png',
    PROJECT_ROOT / 'figures' / f'{prefix}comparison_model_graph.png',
]

for path in figure_paths:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print('Missing file. Run the training cell first.')

## 7. Compare Final Scores in Notebook

This table summarizes the in-memory experiment results from the current run.

In [ ]:
rows = []
for dataset_key, experiment in experiments.items():
    for model_name, result in experiment['metrics'].items():
        rows.append({
            'Dataset': experiment['dataset_name'],
            'Model': model_name,
            'Accuracy': result['accuracy'],
            'Weighted F1': result['f1'],
            'Macro F1': result['macro_f1'],
        })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    rows